<a href="https://colab.research.google.com/github/mlnjsh/Introduction-to-Deep-Learning/blob/main/notebooks/filters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!streamlit run filterrs.py


^C


In [6]:
from dash import Dash, html, dcc, Input, Output
import numpy as np
import plotly.graph_objects as go

# Initialize Dash app
app = Dash(__name__)

# Helper function to generate matrices and plot
def generate_matrices(input_size, padding, kernel_size, stride, kernel_position=(0, 0)):
    # Calculate padded dimensions and output size
    padded_size = input_size + 2 * padding
    output_size = (padded_size - kernel_size) // stride + 1

    # Create input matrix with padding
    input_matrix = np.zeros((padded_size, padded_size))
    if padding > 0:
        input_matrix[padding:padding + input_size, padding:padding + input_size] = 1
    else:
        input_matrix[:input_size, :input_size] = 1

    # Create output matrix
    output_matrix = np.zeros((output_size, output_size))

    # Define kernel bounds for visualization
    start_x, start_y = kernel_position
    kernel_matrix = np.zeros_like(input_matrix)
    kernel_matrix[start_x:start_x + kernel_size, start_y:start_y + kernel_size] = 1

    return input_matrix, kernel_matrix, output_matrix

# Function to plot the matrices
def plot_matrices(input_matrix, kernel_matrix, output_matrix):
    fig = go.Figure()

    # Input matrix
    fig.add_trace(go.Heatmap(
        z=input_matrix,
        colorscale='Greys',
        showscale=False,
        name='Input'
    ))

    # Kernel matrix overlay
    fig.add_trace(go.Heatmap(
        z=kernel_matrix,
        colorscale='Reds',
        showscale=False,
        opacity=0.5,
        name='Kernel'
    ))

    # Update layout
    fig.update_layout(
        title="Input Matrix with Kernel Overlay",
        xaxis=dict(showgrid=False, zeroline=False),
        yaxis=dict(showgrid=False, zeroline=False),
        height=500, width=500
    )

    return fig

# Layout for the Dash app
app.layout = html.Div([
    html.H1("Convolution Visualization with Dash", style={"text-align": "center"}),

    html.Div([
        html.Label("Input Size:"),
        dcc.Slider(id='input-size', min=4, max=10, value=4, marks={i: str(i) for i in range(4, 11)}, step=1),
        
        html.Label("Padding:"),
        dcc.Slider(id='padding', min=0, max=5, value=1, marks={i: str(i) for i in range(6)}, step=1),

        html.Label("Kernel Size:"),
        dcc.Slider(id='kernel-size', min=1, max=5, value=2, marks={i: str(i) for i in range(1, 6)}, step=1),

        html.Label("Stride:"),
        dcc.Slider(id='stride', min=1, max=3, value=1, marks={i: str(i) for i in range(1, 4)}, step=1),

        html.Label("Kernel Position (Row, Column):"),
        dcc.Input(id='kernel-row', type='number', value=0, step=1, placeholder="Row"),
        dcc.Input(id='kernel-col', type='number', value=0, step=1, placeholder="Column"),
    ], style={"padding": "20px"}),

    dcc.Graph(id='matrix-visualization')
])

# Callback to update visualization
@app.callback(
    Output('matrix-visualization', 'figure'),
    [
        Input('input-size', 'value'),
        Input('padding', 'value'),
        Input('kernel-size', 'value'),
        Input('stride', 'value'),
        Input('kernel-row', 'value'),
        Input('kernel-col', 'value')
    ]
)
def update_visualization(input_size, padding, kernel_size, stride, kernel_row, kernel_col):
    # Ensure kernel position is within bounds
    kernel_row = max(0, min(kernel_row, input_size + 2 * padding - kernel_size))
    kernel_col = max(0, min(kernel_col, input_size + 2 * padding - kernel_size))

    # Generate matrices
    input_matrix, kernel_matrix, output_matrix = generate_matrices(
        input_size, padding, kernel_size, stride, kernel_position=(kernel_row, kernel_col)
    )

    # Plot the matrices
    fig = plot_matrices(input_matrix, kernel_matrix, output_matrix)
    return fig

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)
